# Transformer 深入理解：从 Demo 到原理（零基础 + 交互可视化）

面向几乎零基础的读者。风格靠近 [3Blue1Brown 神经网络系列](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)：**先建立画面与直觉，再写少量公式；能拖就拖，能点就点**。

## 你将学会什么

1. Transformer **能做什么**（分类、填空、生成、摘要……）
2. 神经网络与**词向量**的最小工具箱
3. **注意力**在算什么（软查找）
4. 如何用纯 PyTorch 拼一个迷你 GPT，并看它学习

## 怎么用这份 Notebook

1. 用本仓库 Docker 环境打开（见仓库 `README.md`）
2. 右上角 **Trust** 本 notebook，否则交互控件可能不显示
3. 建议 **按 Part 顺序运行**；每个交互单元旁都写了「拖什么 / 看什么」
4. Part 1 / 3 首次运行需要**联网下载**小模型

## 路线图

**魔术（Demo）→ 零件（神经元 / 词向量）→ 组装（Attention / Transformer）→ 自己造一个小模型**


In [1]:
# 环境准备：把 notebooks/ 加入路径，便于 import viz
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
candidates = [
    ROOT,
    ROOT / "notebooks",
    ROOT.parent,
    ROOT.parent / "notebooks",
]
for candidate in candidates:
    if (candidate / "viz").is_dir():
        sys.path.insert(0, str(candidate))
        print("viz path:", candidate)
        break
else:
    raise RuntimeError("找不到 notebooks/viz，请从仓库根目录或 notebooks/ 启动 Jupyter")

import os
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import numpy as np
import torch
import plotly.io as pio

pio.renderers.default = "notebook_connected"

print("numpy", np.__version__)
print("torch", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")


viz path: /home/jovyan/work/notebooks
numpy 2.5.4
torch 2.14.1+cpu
device: cpu


### 参考（可课后看，正文自洽）

- 3Blue1Brown: [Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)、[But what is a GPT? Visual intro to Transformers](https://www.youtube.com/watch?v=wjZofJX0v4M)
- Hugging Face Course Ch.1: [Transformers, what can they do?](https://huggingface.co/learn/nlp-course/chapter1/3)
- [waylandzhang/Transformer-from-scratch](https://github.com/waylandzhang/Transformer-from-scratch)、Karpathy [nanoGPT](https://github.com/karpathy/nanoGPT)
- 原论文: [Attention Is All You Need](https://arxiv.org/abs/1706.03762)


---
# Part 1 — 先看魔术：Transformer 到底有什么用？

先**不拆内部**。选一个任务、改一改输入、点运行，感受同一类模型能覆盖多少 NLP 场景。

| 任务 | 背后在做的事 | 常见架构 |
|------|--------------|----------|
| 情感分析 | 整句 → 一个标签 | Encoder |
| 填空 | 根据上下文猜被遮的词 | Encoder (MLM) |
| NER | 每个词一个标签 | Encoder |
| 文本生成 | 反复预测下一个词 | Decoder |
| 摘要 | 读长文、写短文 | Encoder-Decoder |


In [2]:
from viz import interactive_pipeline_playground

interactive_pipeline_playground()


> **小练习**：在「文本生成」里换不同开头；在「填空」里把 `[MASK]` 放在不同位置。  
> 问自己：模型是在「理解意思」，还是在「匹配训练时见过的统计模式」？（后面会回到这个问题。）


---
# Part 2 — 铺垫：深度学习最小工具箱（3b1b 风格）

要看懂 Transformer，其实只需要几块积木：

1. **神经元**：对输入做加权求和，再用非线性“弯曲”
2. **Softmax**：把一串可正可负的分数，变成非负、加起来为 1 的权重（注意力和“下一个词”都靠它）
3. **损失 + 梯度下降**：用错误当指南针，一点点改参数
4. **词向量**：把词变成空间里的点，让“相近意思”靠得近


## 2.1 一个神经元如何切开平面

一个神经元计算：$y = f(w_1 x_1 + w_2 x_2 + b)$。  
在二维平面上，$w\cdot x + b = 0$ 是一条直线——**决策边界**。

- **拖什么**：`w1` / `w2` / `bias`
- **看什么**：红线如何旋转、平移，把两类点分开


In [3]:
from viz import interactive_neuron_boundary

interactive_neuron_boundary()


## 2.2 Softmax：把分数变成“概率饼”

模型内部常常先算出一串**分数**（logits）：比如“下一个词有多像是 cat / dog / car”，或者“这个 token 该与句子里其余位置有多像”。分数是任意实数，可正可负，加起来也不是 1。**还不能当成概率或权重。**

Softmax 就是把这串分数 $z$ 变成同样长度的向量 $p$：

$$
p_i = \mathrm{softmax}(z)_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

默认温度 $T = 1$。把公式拆开是三步：

1. **除以 $T$**：先把分数差拉大（$T$ 小）或压小（$T$ 大）。
2. **取指数 $e^{z_i/T}$**：正数变大，负数变成很小的正数。例如 $e^{0}=1$，$e^{1}\approx 2.7$，$e^{-1}\approx 0.37$。
3. **除以总和**：每一项变成“占总量的多少”，所以 $p_i>0$ 且 $\sum_i p_i = 1$。

手算一下 $z=(2,1,0)$、$T=1$：

| 选项 | 分数 | $e^{z_i}$ | 占比 |
|------|------|---------|------|
| A | 2 | 7.39 | 7.39 / 11.11 ≈ **0.67** |
| B | 1 | 2.72 | ≈ **0.24** |
| C | 0 | 1.00 | ≈ **0.09** |

A 只比 B 多 1 分，权重却大了将近三倍——指数会**放大差距**。

### 为什么能用它

注意力要拿这些数去**加权平均**若干个 value 向量；语言模型要拿它们当**下一个词的概率**。两处都要求同一件事：每个权重 $\ge 0$，全部加起来等于 1。Softmax 刚好满足，而且还多几条别的替代做不到的性质：

1. **把负分数也变成正权重。** 点积相似度可以是负数。直接拿负数当权重，等于在加权平均里**减掉**一部分内容。$e^{z}$ 永远大于 0。
2. **和为 1，输出尺度稳。** 权重加起来是 1 时，结果是各个向量的凸组合（落在它们围成的区域里）。若只是把分数加起来再除，选项一多、分数一大，输出长度就会乱飘。
3. **谁分高谁权重大，而且可导。** 分数的大小顺序被保留，梯度能传回去改产生这些分数的参数。硬挑最大（argmax）几乎到处梯度为 0，训练不动，也丢掉了“第二名也有点相关”的信息。
4. **差距会被指数放大。** 这就是“软的赢家通吃”：最像的那个占大头，其余仍留一点份额。温度就是这个放大倍数的旋钮。$T\to 0$ 趋近只看最大项（几乎 one-hot）；$T\to\infty$ 趋近大家平分。

所以不能用更简单的替代：

| 做法 | 问题 |
|------|------|
| 分数直接除以总和 | 一旦有负数，“占比”可以为负或大于 1 |
| 每项除以最大值 | 最大项变成 1，但其余项加起来不是 1 |
| argmax / 硬 max | 只留一个赢家，几乎没有梯度 |

实现里还要防溢出：$e^{100}$ 会大到计算机存不下。先减去最大值再取指数，数学上等价（分子分母同乘 $e^{-\max z}$），结果不变。先拖下面的温度，再运行其后那格代码：它会把这张表和“带负数时不能直接除以总和”打出来。

和后文的两处对应：

- **注意力**：对每一行相似度分数做 softmax，得到“这个 token 该看其余位置各多少”。
- **下一个词**：对词表上的分数做 softmax，得到一个概率分布。训练时用交叉熵 $-\log p_{\text{正确词}}$：正确词的概率越接近 1，损失越接近 0。

- **拖什么**：温度 $T$（松手后图才更新）
- **看什么**：$T$ 小，柱子变尖（几乎只信分数最高的那个）；$T$ 大，柱子变平（几乎平分）


In [4]:
from viz import interactive_softmax_temperature

interactive_softmax_temperature([2.0, 1.0, 0.2, -0.5, -1.2])


In [5]:
import numpy as np

z = np.array([2.0, 1.0, 0.0])
z_neg = np.array([2.0, -1.0, 0.5])

def softmax(scores, temperature=1.0):
    x = np.asarray(scores, dtype=float) / temperature
    x = x - x.max()  # 减去最大值，防溢出；分子分母同乘一个常数，结果不变
    e = np.exp(x)
    return e / e.sum()

print("分数 z:", z)
print("exp(z):", np.round(np.exp(z), 3))
print("softmax:", np.round(softmax(z), 3), "和 =", round(float(softmax(z).sum()), 6))
print("T=0.5 更尖:", np.round(softmax(z, 0.5), 3))
print("T=5 更平:", np.round(softmax(z, 5.0), 3))
print("带负数时直接除以总和:", np.round(z_neg / z_neg.sum(), 3), "← 出现了负数，不能当权重")
print("带负数时 softmax:", np.round(softmax(z_neg), 3), "和 =", round(float(softmax(z_neg).sum()), 6))


分数 z: [2. 1. 0.]
exp(z): [7.389 2.718 1.   ]
softmax: [0.665 0.245 0.09 ] 和 = 1.0
T=0.5 更尖: [0.867 0.117 0.016]
T=5 更平: [0.402 0.329 0.269]
带负数时直接除以总和: [ 1.333 -0.667  0.333] ← 出现了负数，不能当权重
带负数时 softmax: [0.786 0.039 0.175] 和 = 1.0


## 2.3 梯度下降：在损失曲面上下山

训练 = 找到让 **loss** 更小的参数。梯度指向“上山”最陡的方向，我们反着走。

分类和“预测下一个词”里，损失通常就是上一节的交叉熵：先 softmax 得到概率 $p$，若正确答案是第 $k$ 项，则 $L=-\log p_k$。$p_k$ 越接近 1，$L$ 越接近 0；若把正确答案压到接近 0，$-\log$ 会变得很大。

- **拖什么**：学习率
- **点什么**：单步 / 自动播放
- **看什么**：红点是否走向谷底；学习率太大可能会跳来跳去


In [6]:
from viz import interactive_gradient_descent

interactive_gradient_descent()


## 2.4 词如何变成数：One-hot vs 词向量

计算机不能直接吃汉字/英文单词，得先变成向量。

- **One-hot**：每个词一根独热轴——稀疏，且任意两词“正交”，看不出 cat 和 dog 更像
- **Embedding（词向量）**：每个词一个稠密小向量——可学习，相似词可靠近


In [7]:
from viz import one_hot_vs_embedding_demo, interactive_embedding_space

one_hot_vs_embedding_demo(["cat", "dog", "car"])
print("\n--- 下面是可旋转的玩具词向量空间 ---")
interactive_embedding_space()


One-hot（稀疏、任意两词正交，看不出相似性）：
  cat  -> [1, 0, 0]
  dog  -> [0, 1, 0]
  car  -> [0, 0, 1]

Embedding（稠密、可学习；相似词可有相近方向）：
  cat  -> [0.17, 0.41, 0.17, -0.65]
  dog  -> [0.45, 0.22, -0.27, 0.29]
  car  -> [0.18, 0.15, 0.01, 0.27]

--- 下面是可旋转的玩具词向量空间 ---


## 2.5 序列为什么难？

- **袋装词（bag of words）**：丢掉顺序 → “狗咬人”和“人咬狗”一样
- **RNN**：按时间一步步读，长程依赖容易丢、训练也慢
- **Transformer 的想法**：让每个位置**直接**看全序列（或过去所有位置），用注意力决定“该看谁、看多少”

这就引出 Part 3。


---
# Part 3 — 桥接：文本如何走进模型

数据流可以记成一句话：

**文本 → 子词 token → 整数 id → 查 embedding 表 →（加上位置）→ 一叠 Transformer 块 → 任务头**


## 3.1 Tokenization：词 / 子词

现代模型很少按“空格分词”，而是用 **BPE / WordPiece** 把稀有词拆成子词，控制词表大小。


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text = "Transformers are amazing!"
encoded = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

print("原文:", text)
print("tokens:", tokens)
print("input_ids:", encoded["input_ids"])
print("注意：[CLS] / [SEP] 是特殊符号；## 前缀表示子词续片。")


Could not reach the Hub ([Errno 99] Cannot assign requested address). Using cached commit hash for 'distilbert-base-uncased'.
'[Errno 99] Cannot assign requested address' thrown while requesting GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
'[Errno 99] Cannot assign requested address' thrown while requesting GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates
Retrying in 2s [Retry 2/5].
'[Errno 99] Cannot assign requested address' thrown while requesting GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates
Retrying in 4s [Retry 3/5].
'[Errno 99] Cannot assign requested address' thrown while requesting GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates
Retrying in 8s [Retry 4/5].


## 3.2 前向一瞥：隐藏状态的形状

对 Encoder 模型，输出常见形状是 `(batch, seq_len, hidden_size)`——每个 token 一个向量。


In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("distilbert-base-uncased")
model.eval()

inputs = tokenizer("The cat sat on the mat", return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print("input_ids:", tuple(inputs["input_ids"].shape), "→ (batch, seq)")
print("last_hidden_state:", tuple(outputs.last_hidden_state.shape), "→ (batch, seq, hidden)")
print("hidden_size =", outputs.last_hidden_state.shape[-1])


## 3.3 第一个 Aha：点选 token，看注意力

注意力 ≈ **软查找**：query 问“我该看谁？”，key 回答相似度，value 提供内容。

下面先用**教学用**的清晰权重练手；再可选加载真实 DistilBERT 的某一层某一头。


In [ ]:
from viz import interactive_attention_explorer

interactive_attention_explorer("The cat sat on the mat")


In [ ]:
# 可选：看真实模型的注意力（首次需下载）
from viz import attention_from_hf_model, interactive_attention_explorer

tokens, weights = attention_from_hf_model(
    "The cat sat on the mat",
    model_name="distilbert-base-uncased",
    layer=0,
    head=0,
)
print("tokens:", tokens)
interactive_attention_explorer(tokens=tokens, weights=weights)


---
# Part 4 — 数学与原理：Attention 在算什么

## 图书馆类比

- **Query（Q）**：我带着的问题
- **Key（K）**：每本书脊上的标签
- **Value（V）**：书里的内容

用 Q 去和所有 K 比相似度，得到权重，再对 V **加权平均**——这就是一次注意力。


## 4.1 Scaled Dot-Product Attention

$$
\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

$QK^\top$ 的每一行是任意实数：可正可负，也不保证和为 1。下一步要拿它们当权重去平均 $V$。这里必须用 softmax，原因就是 Part 2 那几条：权重为正、每一行和为 1、谁更相似谁权重大，而且可导。输出因此是各个 value 的加权平均，不会因为 token 变多就把向量吹大。若改成 argmax，就变成“只抄最像的那一个 value”，梯度几乎传不回去，也丢掉了“几个相关词一起看”。

除以 $\sqrt{d_k}$ 是为了让 softmax 不要太早饱和。若 $q,k$ 的每个分量大致独立、方差约为 1，点积是 $d_k$ 个乘积之和，典型大小随 $\sqrt{d_k}$ 增长。维度一高，分数差被放大，softmax 会挤成几乎 one-hot，后面的层收不到有用梯度。除以 $\sqrt{d_k}$ 把分数拉回大致不随维度漂的尺度。


In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# 小数值例子：3 个 token，d_k=2
Q = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
K = Q.copy()
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])

out, w = scaled_dot_product_attention(Q, K, V)
print("attention weights:\n", np.round(w, 3))
print("output:\n", np.round(out, 3))


## 4.2 因果 Mask：生成时不能偷看未来

Decoder（GPT 系）在预测第 $t$ 个词时，只能看 $\le t$ 的位置。


In [ ]:
from viz import interactive_qk_heatmap

interactive_qk_heatmap(["The", "cat", "sat", "on", "mat"])


## 4.3 温度再练一次

下面这 5 个数当成某个 query 对 5 个 key 的原始分数。温度把 Part 2 的旋钮再拧一次：$T$ 小，几乎只取分数最大的那个 value；$T$ 大，五个 value 被差不多平均。

真实注意力里温度通常固定为 1，尖和平主要由分数本身和 $1/\sqrt{d_k}$ 决定。生成文本时，有的解码器会再把词表 logits 除以一个温度：小于 1 更贪心（总挑概率最高的词），大于 1 更随机。


In [ ]:
from viz import interactive_softmax_temperature

# 假装这是某 query 对 5 个 key 的原始分数
interactive_softmax_temperature([3.0, 1.5, 0.5, 0.0, -1.0])


## 4.4 Multi-Head Attention

多头 = 把 $d_{model}$ 拆成 $h$ 份，每份学一种“关系视角”（句法、指代、位置邻近……），再拼回去投影。

$$
\mathrm{MultiHead}(Q,K,V) = \mathrm{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_h) W^O
$$


In [ ]:
import torch
import torch.nn as nn

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=32, n_head=4):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, causal=False):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        # (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if causal:
            mask = torch.tril(torch.ones(T, T, device=x.device))
            att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), att

mha = MultiHeadSelfAttention()
x = torch.randn(1, 5, 32)
y, att = mha(x, causal=True)
print("output:", tuple(y.shape))
print("attn :", tuple(att.shape), "→ (batch, heads, T, T)")


## 4.5 位置编码：顺序从哪里来？

自注意力本身**置换近似不变**（不看绝对位置）。所以要注入位置信息。

原论文用 **sin / cos** 函数；现代 LLM 还常用 **RoPE** 等（这里只建立 sin/cos 直觉）。


In [ ]:
from viz import interactive_positional_encoding

interactive_positional_encoding(d_model=32, max_len=64)


## 4.6 拼成一个 Block

一个典型 Decoder block（Pre-LN 风格）：

1. LayerNorm → Masked Multi-Head Self-Attention → 残差相加  
2. LayerNorm → Feed-Forward（两层 MLP）→ 残差相加  

残差让深层更容易训练；FFN 在每个位置上独立做非线性变换。

三种注意力：

| 种类 | 谁查谁 | 哪里用 |
|------|--------|--------|
| Self-Attention | 自己看自己 | Encoder / Decoder |
| Masked Self-Attention | 自己看过去 | Decoder |
| Cross-Attention | Decoder query，Encoder key/value | 机器翻译等 Enc-Dec |


---
# Part 5 — 迷你实现：自己训练一个小 GPT

下面用纯 PyTorch（**不用** `nn.Transformer`）训练一个字符级迷你 Decoder。

- 目标：下一个字符预测（Causal LM）
- 观察：loss 是否下降；续写是否从乱码变得略像训练文本
- 这是**教具**，不是生产模型


In [ ]:
from viz import train_mini_gpt_with_plot

model, meta = train_mini_gpt_with_plot(
    steps=300,
    block_size=32,
    batch_size=32,
    lr=3e-3,
    sample_prompt="Attention ",
)
print("vocab size:", len(meta["chars"]))
print("final loss:", round(meta["losses"][-1], 3))


### 和前面的联系

- Part 2 的 **交叉熵 loss + 梯度下降** 正在这里工作
- Part 4 的 **因果 mask 多头注意力** 就是 `MiniGPT` 的核心
- 放大数据、加深层数、换成子词词表——就走向 GPT 系语言模型


---
# Part 6 — 架构族谱与下一步

| 家族 | 结构 | 典型任务 | 代表 |
|------|------|----------|------|
| Encoder-only | 双向自注意力 | 分类、NER、填空 | BERT、DistilBERT |
| Decoder-only | 因果自注意力 | 续写、对话、LLM | GPT、LLaMA |
| Encoder-Decoder | 编码 + 交叉注意 + 解码 | 翻译、摘要 | T5、BART、原论文 Transformer |

## 建议的下一步

1. 重看 3b1b 的 GPT / Transformer 可视化视频，对照本 notebook 的交互件  
2. 读完 HF Course Chapter 1–2，试着 fine-tune 一个小分类器  
3. 跟 [nanoGPT](https://github.com/karpathy/nanoGPT) 或 [Transformer-from-scratch](https://github.com/waylandzhang/Transformer-from-scratch) 做一个稍大的字符/子词模型  
4. 回到 Part 1：再用 pipeline 时，试着说出数据流里每一步的形状

---

**恭喜你走完一遍：从“会用”到“看得出注意力在软查找，生成在戴着因果眼镜预测下一个词”。**  
若某一段仍模糊，优先重做对应的交互单元，而不是死背公式。
